# NHL Home Game Predictor — Model Training

Train Logistic Regression, Random Forest, and XGBoost. Compare performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pipeline.data_loader import load_all_seasons, TRAIN_SEASONS, TEST_SEASON
from pipeline import features, noise_reduction, calibration
from models import train, evaluate

df_all = load_all_seasons()
df_all_fe = features.build_features(df_all)
df_train_fe = noise_reduction.apply(df_all_fe[df_all_fe['season'].isin(TRAIN_SEASONS)].copy())
print('Training shape:', df_train_fe.shape)

In [ ]:
# Prepare X, y using N=20 window features
train_feat_cols = evaluate.get_feature_cols_for_window(20, df_train_fe)
train_feat_cols = [c for c in train_feat_cols if c in df_train_fe.columns]

y_train = df_train_fe['result'].eq('W').astype(int)
X_train = df_train_fe[train_feat_cols].copy()

valid = X_train.notna().all(axis=1)
X_train = X_train[valid]
y_train = y_train[valid]

print(f'X_train: {X_train.shape}, win rate: {y_train.mean():.3f}')

In [ ]:
# Train all models
models = train.train_all(X_train, y_train)

In [ ]:
# Feature importance — Random Forest
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=train_feat_cols).sort_values(ascending=False)
importances.head(20).plot(kind='barh', figsize=(8, 8), title='RF Feature Importances (top 20)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Calibration
cal_split = int(len(X_train) * 0.8)
X_cal, y_cal = X_train.iloc[cal_split:], y_train.iloc[cal_split:]

from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_prob = model.predict_proba(X_cal)[:, 1]
    fop, mpv = calibration_curve(y_cal, y_prob, n_bins=10)
    ax.plot(mpv, fop, 's-', label='Before')

    cal_model = calibration.calibrate(model, X_cal, y_cal)
    y_prob_cal = cal_model.predict_proba(X_cal)[:, 1]
    fop_cal, mpv_cal = calibration_curve(y_cal, y_prob_cal, n_bins=10)
    ax.plot(mpv_cal, fop_cal, 'o--', label='After (Platt)')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_title(name)
    ax.legend()

plt.suptitle('Calibration Curves')
plt.tight_layout()
plt.show()